In [67]:
# --- 0) Imports ---
import os, json, time, re
from pathlib import Path

import json, time
from pathlib import Path
from datetime import datetime, timezone
# Find repo root
def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start).resolve()
    for p in [start] + list(start.parents):
        if (p / ".git").exists() or (p / "README.md").exists():
            return p
    raise RuntimeError(f"Repo root not found from start={start}")

repo_root = find_repo_root()
print("Repo root:", repo_root)

# Load API key
KEYS_PATH = repo_root / "keys.json"
if not KEYS_PATH.exists():
    raise FileNotFoundError(f"keys.json missing: {KEYS_PATH}")

keys = json.loads(KEYS_PATH.read_text())
OPENAI_API_KEY = keys.get("openai")
if not OPENAI_API_KEY:
    raise ValueError("Missing 'openai' key in keys.json")

# Install + init OpenAI
%pip install -q openai
from openai import OpenAI
client = OpenAI(api_key=OPENAI_API_KEY)

print("OpenAI client initialized.")


Repo root: /var/lib/jenkins/ici01/ici01_PublicOpinionAnalyzer
Note: you may need to restart the kernel to use updated packages.
OpenAI client initialized.


In [68]:
# ------------------------------------------------------------------
# Dataset root
# ------------------------------------------------------------------
dataset_root = repo_root / "datasets" / "yt_factlink"
print("Dataset root:", dataset_root)

DATASET_ID = "yt_factlink"

# ------------------------------------------------------------------
# Evaluation split
# ------------------------------------------------------------------
SPLIT_NAME = "split_v1_seed42"
SPLIT_DIR = dataset_root / "01_conversion" / "03_splits" / SPLIT_NAME
EVAL_JSONL = SPLIT_DIR / "test.data.jsonl"
assert SPLIT_DIR.exists(), f"Missing split folder: {SPLIT_DIR}"
assert EVAL_JSONL.exists(), f"Missing test split: {EVAL_JSONL}"

print("Evaluation mode: TEST SPLIT")
print("Eval file:", EVAL_JSONL)

# ------------------------------------------------------------------
# Provider + model
# ------------------------------------------------------------------
PROVIDER = "openai"
MODEL_NAME = "gpt-4.1"
SUFFIX_TAG = "single_class"

# ------------------------------------------------------------------
# Single-class prompt definitions
# ------------------------------------------------------------------
PROMPT_BASE_DIR = dataset_root / "02_prompts" / "single_class"
CLASSIFICATIONS = [
    {"label": "C1", "name": "Taiwanese Nationalism"},
    {"label": "C2", "name": "Antipathy Toward China"},
    {"label": "C3", "name": "Antipathy Toward Taiwan"},
    {"label": "C4", "name": "Questioning Video Authenticity"},
    {"label": "C6", "name": "Coordinated Inauthentic Behavior"},
]

for entry in CLASSIFICATIONS:
    label_dir = PROMPT_BASE_DIR / entry["label"]
    entry["system_path"] = label_dir / "system.txt"
    entry["user_path"] = label_dir / "user.txt"
    assert entry["system_path"].exists(), f"Missing system prompt for {entry['label']}: {entry['system_path']}"
    assert entry["user_path"].exists(), f"Missing user prompt for {entry['label']}: {entry['user_path']}"

print("Loaded single-class prompts from:", PROMPT_BASE_DIR)

# ------------------------------------------------------------------
# Helper
# ------------------------------------------------------------------
def slugify(s: str) -> str:
    s = re.sub(r"[^a-zA-Z0-9._-]+", "-", s)
    s = re.sub(r"-{2,}", "-", s)
    return s.strip("-")

NOTEBOOK_DIR = Path.cwd()
MODEL_DIR = NOTEBOOK_DIR / slugify(MODEL_NAME)
MODEL_DIR.mkdir(exist_ok=True)
(MODEL_DIR / ".gitkeep").write_text("")

RUNS_ROOT = MODEL_DIR / "runs"
RUNS_ROOT.mkdir(exist_ok=True)
(RUNS_ROOT / ".gitkeep").write_text("")

timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H%M%SZ")
run_name = "_".join([
    timestamp,
    slugify(PROVIDER),
    slugify(MODEL_NAME),
    slugify(SUFFIX_TAG),
    slugify(SPLIT_NAME),
])
RUN_DIR = RUNS_ROOT / run_name
RUN_DIR.mkdir(parents=True, exist_ok=False)

print("Notebook directory:", NOTEBOOK_DIR.resolve())
print("Model directory   :", MODEL_DIR.resolve())
print("Runs root         :", RUNS_ROOT.resolve())
print("Run directory     :", RUN_DIR.resolve())

print("=== Configuration OK ===")
print("Provider   :", PROVIDER)
print("Model      :", MODEL_NAME)
print("Prompt tag :", SUFFIX_TAG)
print("Split      :", SPLIT_NAME)
print("Run dir    :", RUN_DIR)


Dataset root: /var/lib/jenkins/ici01/ici01_PublicOpinionAnalyzer/datasets/yt_factlink
Evaluation mode: TEST SPLIT
Eval file: /var/lib/jenkins/ici01/ici01_PublicOpinionAnalyzer/datasets/yt_factlink/01_conversion/03_splits/split_v1_seed42/test.data.jsonl
Loaded single-class prompts from: /var/lib/jenkins/ici01/ici01_PublicOpinionAnalyzer/datasets/yt_factlink/02_prompts/single_class
Notebook directory: /var/lib/jenkins/ici01/ici01_PublicOpinionAnalyzer/datasets/yt_factlink/03_accuracy_testing/single_class/openai
Model directory   : /var/lib/jenkins/ici01/ici01_PublicOpinionAnalyzer/datasets/yt_factlink/03_accuracy_testing/single_class/openai/gpt-4.1
Runs root         : /var/lib/jenkins/ici01/ici01_PublicOpinionAnalyzer/datasets/yt_factlink/03_accuracy_testing/single_class/openai/gpt-4.1/runs
Run directory     : /var/lib/jenkins/ici01/ici01_PublicOpinionAnalyzer/datasets/yt_factlink/03_accuracy_testing/single_class/openai/gpt-4.1/runs/2025-11-27T113232Z_openai_gpt-4.1_single_class_split_v1

In [69]:
def read_jsonl(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

test_rows = read_jsonl(EVAL_JSONL)
print("Test examples:", len(test_rows))

for entry in CLASSIFICATIONS:
    entry["system_template"] = entry["system_path"].read_text(encoding="utf-8")
    entry["user_template"] = entry["user_path"].read_text(encoding="utf-8")
    print(f"Loaded prompts for {entry['label']} from {entry['system_path'].parent}")


Test examples: 77
Loaded prompts for C1 from /var/lib/jenkins/ici01/ici01_PublicOpinionAnalyzer/datasets/yt_factlink/02_prompts/single_class/C1
Loaded prompts for C2 from /var/lib/jenkins/ici01/ici01_PublicOpinionAnalyzer/datasets/yt_factlink/02_prompts/single_class/C2
Loaded prompts for C3 from /var/lib/jenkins/ici01/ici01_PublicOpinionAnalyzer/datasets/yt_factlink/02_prompts/single_class/C3
Loaded prompts for C4 from /var/lib/jenkins/ici01/ici01_PublicOpinionAnalyzer/datasets/yt_factlink/02_prompts/single_class/C4
Loaded prompts for C6 from /var/lib/jenkins/ici01/ici01_PublicOpinionAnalyzer/datasets/yt_factlink/02_prompts/single_class/C6


In [70]:
label_runs = {}
relative_eval_path = str(EVAL_JSONL.relative_to(repo_root))

for entry in CLASSIFICATIONS:
    label = entry["label"]
    label_dir = RUN_DIR / label
    inputs_dir = label_dir / "01_inputs"
    outputs_dir = label_dir / "02_outputs"
    snapshots_dir = label_dir / "03_snapshots"

    for d in (label_dir, inputs_dir, outputs_dir, snapshots_dir):
        d.mkdir(parents=True, exist_ok=True)
        (d / ".gitkeep").write_text("")

    (inputs_dir / "system.txt").write_text(entry["system_template"], encoding="utf-8")
    (inputs_dir / "user.txt").write_text(entry["user_template"], encoding="utf-8")
    (inputs_dir / "eval_file.txt").write_text(relative_eval_path, encoding="utf-8")

    label_runs[label] = {
        "entry": entry,
        "dir": label_dir,
        "inputs_dir": inputs_dir,
        "outputs_dir": outputs_dir,
        "snapshots_dir": snapshots_dir,
        "predictions": [],
    }

print("Prepared single-class run folders:")
for label, cfg in label_runs.items():
    print(f"  {label}: {cfg['dir']}")



Prepared single-class run folders:
  C1: /var/lib/jenkins/ici01/ici01_PublicOpinionAnalyzer/datasets/yt_factlink/03_accuracy_testing/single_class/openai/gpt-4.1/runs/2025-11-27T113232Z_openai_gpt-4.1_single_class_split_v1_seed42/C1
  C2: /var/lib/jenkins/ici01/ici01_PublicOpinionAnalyzer/datasets/yt_factlink/03_accuracy_testing/single_class/openai/gpt-4.1/runs/2025-11-27T113232Z_openai_gpt-4.1_single_class_split_v1_seed42/C2
  C3: /var/lib/jenkins/ici01/ici01_PublicOpinionAnalyzer/datasets/yt_factlink/03_accuracy_testing/single_class/openai/gpt-4.1/runs/2025-11-27T113232Z_openai_gpt-4.1_single_class_split_v1_seed42/C3
  C4: /var/lib/jenkins/ici01/ici01_PublicOpinionAnalyzer/datasets/yt_factlink/03_accuracy_testing/single_class/openai/gpt-4.1/runs/2025-11-27T113232Z_openai_gpt-4.1_single_class_split_v1_seed42/C4
  C6: /var/lib/jenkins/ici01/ici01_PublicOpinionAnalyzer/datasets/yt_factlink/03_accuracy_testing/single_class/openai/gpt-4.1/runs/2025-11-27T113232Z_openai_gpt-4.1_single_class

In [71]:
def build_user_message(template: str, row: dict) -> str:
    return (template
            .replace("[VIDEOTITLE]", row.get("VIDEOTITEL",""))
            .replace("[VIDEOCONTEXT]", row.get("VIDEOCONTEXT",""))
            .replace("[TARGETCOMMENT]", row.get("TARGETCOMMENT","")))

# preview one built message (C1 as example)
print(build_user_message(CLASSIFICATIONS[0]["user_template"], test_rows[0])[:4000])


Input
Title:
史上首次！南非跟隨北京羞辱台灣，竟遭「晶片斷供」48小時後緊急求饒！北京當場看傻！

Video Context:
In September 2025, South Africa, bowing to Beijing's influence, demanded Taiwan downgrade its official diplomatic mission. Taiwan responded to this humiliation with an unprecedented and bold countermeasure: an export embargo on 47 critical high-tech products, including advanced semiconductors, to South Africa. This strategic move instantly exposed South Africa's deep structural dependence on Taiwan's technology, causing immediate domestic political fallout in Pretoria. The crisis forced South Africa to urgently seek dialogue with Taiwan just 48 hours after the embargo was announced.

This event signals a fundamental shift in Taiwan's international posture, transforming its "silicon shield" into a "silicon sword." Taiwan's assertiveness is underpinned by its booming capital markets and its dominant role in the global high-tech supply chain, especially in AI-related chips. The incident revealed a key strategic vulnera

In [72]:
from typing import Dict
from pydantic import create_model

label_parsers: Dict[str, type] = {}

def get_label_parser(label: str):
    if label not in label_parsers:
        label_parsers[label] = create_model(f"{label}Output", **{label: (int, ...)})
    return label_parsers[label]

def call_model_single(label: str, system_prompt: str, user_message: str, temperature: float = 0.0):
    parser = get_label_parser(label)
    resp = client.responses.parse(
        model=MODEL_NAME,
        input=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message},
        ],
        text_format=parser,
        temperature=temperature,
    )
    parsed = resp.output_parsed
    value = int(getattr(parsed, label))
    return value, parsed.model_dump_json()

In [73]:
from IPython.display import clear_output

for label, cfg in label_runs.items():
    entry = cfg["entry"]
    predictions = []
    failures = 0

    for i, row in enumerate(test_rows):
        user_msg = build_user_message(entry["user_template"], row)
        try:
            value, raw_output = call_model_single(label, entry["system_template"], user_msg)
            print(value, raw_output)
        except Exception as e:
            failures += 1
            value = 0
            raw_output = f"<ERROR: {e}>"
            print(e)
            break

        gold_value = int(row["labels"].get(label, 0))
        predictions.append({
            "pred": {label: value},
            "gold": {label: gold_value},
            "raw_output": raw_output,
            "comment": row.get("TARGETCOMMENT", ""),
        })

        clear_output(wait=True)
        print(f"[{label}] {i+1}/{len(test_rows)}")
        print(row.get("TARGETCOMMENT"))
        print("GOLD:", gold_value, "PRED:", value)

    cfg["predictions"] = predictions
    cfg["failures"] = failures
    print(f"[{label}] Done. Total failures: {failures}")


[C6] 77/77
台灣的交通罰單是隨車。日本是隨人，照相舉發須能看清駕駛臉，才能開罰。
GOLD: 0 PRED: 0
[C6] Done. Total failures: 0


In [74]:
def compute_binary_metrics(predictions, label):
    counts = {"tp":0,"fp":0,"tn":0,"fn":0}
    for ex in predictions:
        gold = int(ex["gold"].get(label, 0))
        pred = int(ex["pred"].get(label, 0))
        if pred==1 and gold==1:
            counts["tp"] += 1
        elif pred==1 and gold==0:
            counts["fp"] += 1
        elif pred==0 and gold==0:
            counts["tn"] += 1
        elif pred==0 and gold==1:
            counts["fn"] += 1
    tp, fp, tn, fn = counts["tp"], counts["fp"], counts["tn"], counts["fn"]
    prec = tp/(tp+fp) if (tp+fp)>0 else 0.0
    rec  = tp/(tp+fn) if (tp+fn)>0 else 0.0
    f1   = (2*prec*rec/(prec+rec)) if (prec+rec)>0 else 0.0
    acc  = (tp+tn)/(tp+tn+fp+fn) if (tp+tn+fp+fn)>0 else 0.0
    metrics = {"precision":prec,"recall":rec,"f1":f1,"accuracy":acc, **counts}
    return metrics

metrics_by_label = {}
for label, cfg in label_runs.items():
    metrics = compute_binary_metrics(cfg["predictions"], label)
    cfg["metrics"] = metrics
    metrics_by_label[label] = {
        "metrics": metrics,
        "failures": cfg["failures"],
    }

metrics_by_label


{'C1': {'metrics': {'precision': 0.7894736842105263,
   'recall': 0.8108108108108109,
   'f1': 0.8,
   'accuracy': 0.8051948051948052,
   'tp': 30,
   'fp': 8,
   'tn': 32,
   'fn': 7},
  'failures': 0},
 'C2': {'metrics': {'precision': 0.6521739130434783,
   'recall': 0.8823529411764706,
   'f1': 0.75,
   'accuracy': 0.8701298701298701,
   'tp': 15,
   'fp': 8,
   'tn': 52,
   'fn': 2},
  'failures': 0},
 'C3': {'metrics': {'precision': 0.375,
   'recall': 0.42857142857142855,
   'f1': 0.39999999999999997,
   'accuracy': 0.8831168831168831,
   'tp': 3,
   'fp': 5,
   'tn': 65,
   'fn': 4},
  'failures': 0},
 'C4': {'metrics': {'precision': 0.3,
   'recall': 0.75,
   'f1': 0.4285714285714285,
   'accuracy': 0.8961038961038961,
   'tp': 3,
   'fp': 7,
   'tn': 66,
   'fn': 1},
  'failures': 0},
 'C6': {'metrics': {'precision': 0.0,
   'recall': 0.0,
   'f1': 0.0,
   'accuracy': 0.987012987012987,
   'tp': 0,
   'fp': 1,
   'tn': 76,
   'fn': 0},
  'failures': 0}}

In [75]:
import json, shutil
from pathlib import Path

NOTEBOOK_PATH = Path("run_accuracy.ipynb")
overall_summary = {}

for label, cfg in label_runs.items():
    outputs_dir = cfg["outputs_dir"]
    snapshots_dir = cfg["snapshots_dir"]
    outputs_dir.mkdir(exist_ok=True)
    snapshots_dir.mkdir(exist_ok=True)

    run_config = {
        "dataset_id": DATASET_ID,
        "experiment_type": "single_class",
        "label": label,
        "label_name": cfg["entry"]["name"],
        "provider": PROVIDER,
        "model_name": MODEL_NAME,
        "data_jsonl": relative_eval_path,
        "prompt_dir": str(cfg["entry"]["system_path"].parent.relative_to(repo_root)),
        "timestamp": timestamp,
    }
    (outputs_dir / "run_config.json").write_text(
        json.dumps(run_config, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    with (outputs_dir / "preds.jsonl").open("w", encoding="utf-8") as f:
        for ex in cfg["predictions"]:
            f.write(json.dumps(ex, ensure_ascii=False) + " ")

    metrics_payload = {
        "failures": cfg["failures"],
        "metrics": cfg["metrics"],
    }
    (outputs_dir / "metrics.json").write_text(
        json.dumps(metrics_payload, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    if NOTEBOOK_PATH.exists():
        shutil.copy2(NOTEBOOK_PATH, snapshots_dir / NOTEBOOK_PATH.name)
        print(f"[{label}] Notebook snapshot saved to {(snapshots_dir / NOTEBOOK_PATH.name).resolve()}")
    else:
        print(f"[{label}] WARNING: notebook file not found; snapshot skipped.")

    print(f"[{label}] Saved outputs to {outputs_dir.resolve()}")
    overall_summary[label] = metrics_payload

(RUN_DIR / "overall_summary.json").write_text(
    json.dumps(overall_summary, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("Single-class accuracy runs complete. Summary saved to:", RUN_DIR / "overall_summary.json")


[C1] Notebook snapshot saved to /var/lib/jenkins/ici01/ici01_PublicOpinionAnalyzer/datasets/yt_factlink/03_accuracy_testing/single_class/openai/gpt-4.1/runs/2025-11-27T113232Z_openai_gpt-4.1_single_class_split_v1_seed42/C1/03_snapshots/run_accuracy.ipynb
[C1] Saved outputs to /var/lib/jenkins/ici01/ici01_PublicOpinionAnalyzer/datasets/yt_factlink/03_accuracy_testing/single_class/openai/gpt-4.1/runs/2025-11-27T113232Z_openai_gpt-4.1_single_class_split_v1_seed42/C1/02_outputs
[C2] Notebook snapshot saved to /var/lib/jenkins/ici01/ici01_PublicOpinionAnalyzer/datasets/yt_factlink/03_accuracy_testing/single_class/openai/gpt-4.1/runs/2025-11-27T113232Z_openai_gpt-4.1_single_class_split_v1_seed42/C2/03_snapshots/run_accuracy.ipynb
[C2] Saved outputs to /var/lib/jenkins/ici01/ici01_PublicOpinionAnalyzer/datasets/yt_factlink/03_accuracy_testing/single_class/openai/gpt-4.1/runs/2025-11-27T113232Z_openai_gpt-4.1_single_class_split_v1_seed42/C2/02_outputs
[C3] Notebook snapshot saved to /var/lib/j